# PREPARATION DU DATASET

##### Import des librairies :

In [103]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report)
from sklearn.pipeline import make_pipeline

##### Chargement des données d'entrainement et de test brutes :

In [104]:
data_train = pd.read_csv("train.csv")
data_test = pd.read_csv("test.csv")

In [105]:
data_train["state_code"].unique() # permet de classer les variables
data_train["county_code"].unique() # permet de classer les variables

array([ 6055., 37113., 51087., ..., 48075., 19185., 48275.])

### Variables continues :

`derived_msa-md`, `loan_amount`, `loan_to_value_ratio`, `loan_term`, `property_value`, `income`, `tract_population`, `tract_minority_population_percent`, `ffiec_msa_md_median_family_income`, `tract_to_msa_income_percentage`, `tract_owner_occupied_units`, `tract_one_to_four_family_homes`, `tract_median_age_of_housing_units`, `county_code`

On doit remplir les valeurs manquantes pour les variables suivantes : `loan_to_value_ratio`, `property_value`, `income`, `loan_term`, `county_code`

In [106]:
# 2) Pour chaque colonne numérique : conversion, indicateur de manquant, médiane
num_cols = ["income", "property_value", "loan_term", "loan_to_value_ratio", "county_code"]

for col in num_cols:
    data_train[col + "_missing"] = data_train[col].isnull().astype(int)  # 1) on note les manquants
    data_train[col] = data_train[col].fillna(data_train[col].median())   # 2) on remplit

for col in num_cols:
    data_test[col + "_missing"] = data_test[col].isnull().astype(int) 
    data_test[col] = data_test[col].fillna(data_train[col].median())   

### Variables catégorielles :

Il y a des variables ordinales, pour lesquelles l'ordre est important : `applicant_age`, `co-applicant_age`, `debt_to_income_ratio`

Il y a des variables nominales (pas forcément textuelles !), pour lesquelles l'ordre ne compte pas : `state_code`, `conforming_loan_limit`, `derived_loan_product_type`, `derived_dwelling_category`,  `loan_type`, `lien_status`, `preapproval`, `reverse_mortgage`, `open-end_line_of_credit`, `business_or_commercial_purpose`, `negative_amortization`, `interest_only_payment`, `balloon_payment`, `other_nonamortizing_features`, `construction_method`, `occupancy_type`, `manufactured_home_secured_property_type`, `manufactured_home_land_property_interest`, `total_units`, `applicant_credit_score_type`, `co-applicant_credit_score_type`, `derived_ethnicity`, `derived_race`, `derived_sex`, `applicant_ethnicity-1`, `co-applicant_ethnicity-1`, `applicant_ethnicity_observed`, `co-applicant_ethnicity_observed`, `applicant_race-1`, `co-applicant_race-1`, `applicant_race_observed`, `co-applicant_race_observed`, `applicant_sex`, `co-applicant_sex`, `applicant_sex_observed`, `co-applicant_sex_observed`, `submission_of_application`, `initially_payable_to_institution`, `aus-1`

Ces deux types de variables ne se traitent pas avec le même encodeur. On sépare les deux groupes :

In [91]:
print(data_train["applicant_age"].unique())
print(data_train["co-applicant_age"].unique()) 

['45-54' '65-74' '25-34' '35-44' '>74' '<25' '55-64' nan]
['45-54' nan '35-44' '25-34' '65-74' '55-64' '<25' '>74']


##### Variables catégorielles ordinales :

In [107]:
ordre_age = {"<25": 25, "25-34": 30, "35-44": 40, "45-54": 50,
         "55-64": 60, "65-74": 70, ">74": 74}

data_train["applicant_age"] = data_train["applicant_age"].map(ordre_age)
data_train["co-applicant_age"] = data_train["co-applicant_age"].map(ordre_age)
data_test["applicant_age"] = data_test["applicant_age"].map(ordre_age)
data_test["co-applicant_age"] = data_test["co-applicant_age"].map(ordre_age)

print(data_train["applicant_age"].unique()) # vérification

[50. 70. 30. 40. 74. 25. 60. nan]


In [108]:
print(data_train["debt_to_income_ratio"].unique())

['20%-<30%' '49' '44' '<20%' '30%-<36%' '50%-60%' '47' '36' '39' '>60%'
 '46' '45' '48' nan '43' '37' '40' '41' '42' '38']


In [109]:
ordre_debt = {"<20%": 20, "20%-<30%": 25, "30%-<36%": 33, "36": 36, "37": 37, "38": 38,
              "39": 39, "40": 40, "41": 41, "42": 42, "43": 43, "44": 44, "45": 45, "46": 46,
              "47": 47, "48": 48, "49": 49, "50%-60%": 55, ">60%": 60}

data_train["debt_to_income_ratio"] = data_train["debt_to_income_ratio"].map(ordre_debt)
data_test["debt_to_income_ratio"] = data_test["debt_to_income_ratio"].map(ordre_debt)

print(data_train["debt_to_income_ratio"].unique()) # vérification

[25. 49. 44. 20. 33. 55. 47. 36. 39. 60. 46. 45. 48. nan 43. 37. 40. 41.
 42. 38.]


In [110]:
ord_cols = ["applicant_age", "co-applicant_age", "debt_to_income_ratio"]

for df in (data_train, data_test):
    df[ord_cols] = df[ord_cols].fillna(-2).astype(int) #on dégage les NaN

In [111]:
cols_nom = [
    "state_code", "conforming_loan_limit",
    "derived_loan_product_type", "derived_dwelling_category",
    "loan_type", "lien_status", "preapproval", "reverse_mortgage",
    "open-end_line_of_credit", "business_or_commercial_purpose",
    "negative_amortization", "interest_only_payment", "balloon_payment",
    "other_nonamortizing_features", "construction_method", "occupancy_type",
    "manufactured_home_secured_property_type",
    "manufactured_home_land_property_interest", "total_units",
    "applicant_credit_score_type", "co-applicant_credit_score_type",
    "derived_ethnicity", "derived_race", "derived_sex",
    "applicant_ethnicity-1", "co-applicant_ethnicity-1",
    "applicant_ethnicity_observed", "co-applicant_ethnicity_observed",
    "applicant_race-1", "co-applicant_race-1",
    "applicant_race_observed", "co-applicant_race_observed",
    "applicant_sex", "co-applicant_sex",
    "applicant_sex_observed", "co-applicant_sex_observed",
    "submission_of_application", "initially_payable_to_institution",
    "aus-1",
]

# Ordre alphabétique par défaut de OrdinalEncoder
enc_nom = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1, encoded_missing_value=-2) # je mets un code (-2) pour les NaN au passage
data_train[cols_nom] = enc_nom.fit_transform(data_train[cols_nom])
data_test[cols_nom] = enc_nom.transform(data_test[cols_nom])

In [80]:
print(data_train["balloon_payment"].unique()) # vérification

[1. 0. 2.]


In [112]:
print(data_train.dtypes.value_counts())  # doit n'afficher que int64 / float64

float64    46
int64      17
Name: count, dtype: int64


### Vérifications :

In [113]:
data_train.isnull().sum()

id                             0
loan_amount                    0
loan_to_value_ratio            0
property_value                 0
income                         0
                              ..
income_missing                 0
property_value_missing         0
loan_term_missing              0
loan_to_value_ratio_missing    0
county_code_missing            0
Length: 63, dtype: int64

In [114]:
data_test.isnull().sum()

id                             0
loan_amount                    0
loan_to_value_ratio            0
property_value                 0
income                         0
                              ..
income_missing                 0
property_value_missing         0
loan_term_missing              0
loan_to_value_ratio_missing    0
county_code_missing            0
Length: 62, dtype: int64

##### On enregistre ces nouveaux datasets pour éviter de relancer tote la préparation à chaque fois :

In [115]:
data_train.to_csv("data_train_clean.csv", index=True)
data_test.to_csv("data_test_clean.csv", index=True)